In [22]:
import h5py
import numpy as np
## to see the structure of the hdf5 file and what keys are available
path = "./dataset/ICs_sampling_0.hdf5"
with h5py.File(path, "r") as f:
    def show(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"[DATASET] {name} shape={obj.shape} dtype={obj.dtype}")
        else:
            print(f"[GROUP] {name}")
    f.visititems(show)

[DATASET] APs shape=(40000, 2) dtype=float64
[DATASET] LANs shape=(40000, 2) dtype=float64
[DATASET] aratios shape=(40000, 1) dtype=float64
[DATASET] e_uvecs shape=(40000, 2, 3) dtype=float64
[DATASET] eccs shape=(40000, 2) dtype=float64
[DATASET] h_uvecs shape=(40000, 2, 3) dtype=float64
[DATASET] incs shape=(40000, 2) dtype=float64
[DATASET] masses shape=(40000, 3) dtype=float64
[DATASET] mratios shape=(40000, 2) dtype=float64
[DATASET] periods shape=(40000, 2) dtype=float64
[DATASET] positions shape=(40000, 3, 3) dtype=float64
[DATASET] q_uvecs shape=(40000, 2, 3) dtype=float64
[DATASET] rel_positions shape=(40000, 2, 3) dtype=float64
[DATASET] rel_velocities shape=(40000, 2, 3) dtype=float64
[DATASET] smaxes shape=(40000, 2) dtype=float64
[DATASET] tAnos shape=(40000, 2) dtype=float64
[DATASET] velocities shape=(40000, 3, 3) dtype=float64


In [23]:
# To regroup the data into tokens of size 8, we can reshape the arrays accordingly. 
input_path = "./dataset/ICs_sampling_0.hdf5"
output_path = "./dataset/ICs_sampling_0_groupof8.hdf5"

token_size = 8
tokenized = {}
features = ["aratios", "mratios","eccs","incs","APs","LANs"]

with h5py.File(input_path, "r") as fin:
    for name, obj in fin.items():
        if not isinstance(obj, h5py.Dataset):
            continue
        if not name in features:
            continue
        x = obj[:]
        n = (x.shape[0] // token_size) * token_size
        x_cut = x[:n]
        x_tok = x_cut.reshape(-1, token_size, *x.shape[1:])
        tokenized[name] = x_tok
        print(f"{name}: {x.shape} -> {x_tok.shape}")
        
print("Token regroup done")
# print(tokenized["APs"])

APs: (40000, 2) -> (5000, 8, 2)
LANs: (40000, 2) -> (5000, 8, 2)
aratios: (40000, 1) -> (5000, 8, 1)
eccs: (40000, 2) -> (5000, 8, 2)
incs: (40000, 2) -> (5000, 8, 2)
mratios: (40000, 2) -> (5000, 8, 2)
Token regroup done


In [24]:
import numpy as np

def sincos(x: np.ndarray) -> np.ndarray:
    """x: (..., k) angles in radians -> (..., 2k) = [sin(x), cos(x)]"""
    x = x.astype(np.float32)
    return np.concatenate([np.sin(x), np.cos(x)], axis=-1)

tokenized["APs"]  = sincos(tokenized["APs"])    # (5000, 8, 2) -> (5000, 8, 4)
tokenized["LANs"] = sincos(tokenized["LANs"])   # (5000, 8, 2) -> (5000, 8, 4)
tokenized["incs"] = sincos(tokenized["incs"])   # (5000, 8, 2) -> (5000, 8, 4)

print("APs :", tokenized["APs"].shape)
print("LANs:", tokenized["LANs"].shape)
print("incs:", tokenized["incs"].shape)
print("write the data into the save file ready for the folloing")
with h5py.File(output_path, "w") as fout:
    for name, arr in tokenized.items():
        fout.create_dataset(name, data=arr)
        print(f"{name}: {arr.shape}")

APs : (5000, 8, 4)
LANs: (5000, 8, 4)
incs: (5000, 8, 4)
write the data into the save file ready for the folloing
APs: (5000, 8, 4)
LANs: (5000, 8, 4)
aratios: (5000, 8, 1)
eccs: (5000, 8, 2)
incs: (5000, 8, 4)
mratios: (5000, 8, 2)


In [25]:
import pandas as pd
import numpy as np
import torch

df = pd.read_csv("dataset/completed_systems_sampling_0.txt", sep=r"\s+", header=None,
                 names=["index","num_batch","time_unbound","ignore_other_than-1","dignosis"])

# datatype
df = df.astype({"index":"int64","num_batch":"int64","time_unbound":"float32","ignore_other_than-1":"float32","dignosis":"int64"})
df = df.sort_values("index")  # sort by index column
df["label"] = np.where(df["time_unbound"] >= 5000, "unbound", "bound")  # create binary label based on time_unbound
df["group_8"] = np.arange(len(df)) // 8
prob_each_group = (
    df.groupby("group_8")["label"]
      .apply(lambda x: (x == "unbound").mean())   
      .reset_index(name="prob_unbound")
)
print(df.columns)
print(df.head(10))
print(prob_each_group)
#tensor = torch.from_numpy(arr)  
#print(tensor.shape, tensor.dtype)


Index(['index', 'num_batch', 'time_unbound', 'ignore_other_than-1', 'dignosis',
       'label', 'group_8'],
      dtype='str')
      index  num_batch  time_unbound  ignore_other_than-1  dignosis    label  \
2394      0          0        5000.0            65.464226         0  unbound   
2309      1          0        5000.0            63.461426         0  unbound   
2320      2          0        5000.0            63.699547         0  unbound   
2316      3          0        5000.0            63.610538         0  unbound   
2339      4          0        5000.0            64.137978         0  unbound   
2322      5          0        5000.0            63.498493         0  unbound   
2361      6          0        5000.0            64.632912         0  unbound   
2304      7          0        5000.0            63.364311         0  unbound   
166       8          0          55.0             7.035079         0    bound   
338       9          0          45.0            11.731630         0    bo

In [46]:
import h5py
import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader

# -----------------------------
# 1) Y: prob_unbound per group
# -----------------------------
def build_prob_unbound_labels(
    labels_path="dataset/completed_systems_sampling_0.txt",
    token_size=8,
    threshold_time=5000.0,
):
    df = pd.read_csv(
        labels_path, sep=r"\s+", header=None,
        names=["index","num_batch","time_unbound","ignore_other_than-1","dignosis"]
    )
    df = df.astype({
        "index":"int64","num_batch":"int64","time_unbound":"float32",
        "ignore_other_than-1":"float32","dignosis":"int64"
    })
    df = df.sort_values("index").reset_index(drop=True)

    n = (len(df) // token_size) * token_size
    df = df.iloc[:n].copy()

    df["label"] = (df["time_unbound"] >= threshold_time).astype(np.float32)
    df["group_8"] = np.arange(len(df)) // token_size

    y_prob = df.groupby("group_8")["label"].mean().to_numpy(dtype=np.float32)  # (N_groups,)
    return y_prob


# -----------------------------
# 2) X: take only 1 IC per group
#    from (N, 8, ...) -> (N, ...)
# -----------------------------
def load_tokenized_X_take_one(
    h5_path="dataset/ICs_sampling_0_groupof8.hdf5",
    keys=None,                 # None = all datasets
    take_token_idx=0,          # take the 0-th IC in each group
    dtype=np.float32,
):
    with h5py.File(h5_path, "r") as f:
        all_keys = [k for k in f.keys() if isinstance(f[k], h5py.Dataset)]
        if keys is None:
            keys = all_keys
        else:
            missing = [k for k in keys if k not in f]
            if missing:
                raise KeyError(f"Missing keys in hdf5: {missing}. Available: {all_keys}")

        X_parts = []
        N = None
        for k in keys:
            arr = f[k][:]
            # expect (N, 8, ...)
            if arr.ndim < 2:
                raise ValueError(f"{k} has shape {arr.shape}, expected at least 2D (N,8,...)")
            if N is None:
                N = arr.shape[0]
            elif arr.shape[0] != N:
                raise ValueError(f"{k} has N={arr.shape[0]} but expected N={N}")
            if arr.shape[1] <= take_token_idx:
                raise ValueError(f"{k} second dim is {arr.shape[1]}, cannot take idx={take_token_idx}")

            one = arr[:, take_token_idx, ...].astype(dtype, copy=False)   # (N, ...)
            one_flat = one.reshape(N, -1)                                 # (N, Dk)
            X_parts.append(one_flat)

        X = np.concatenate(X_parts, axis=1)  # (N, D_total)
    return X, keys


def train_test_split_groups(X, y_prob, test_ratio=0.2, seed=42, stratify=True):
    assert X.shape[0] == y_prob.shape[0]
    N = X.shape[0]
    rng = np.random.default_rng(seed)
    idx = np.arange(N)

    if not stratify:
        rng.shuffle(idx)
        n_test = int(round(N * test_ratio))
        return idx[n_test:], idx[:n_test]

    # approximate stratify for continuous y_prob: {0, (0,1), 1}
    bins = np.where(y_prob == 0.0, 0, np.where(y_prob == 1.0, 2, 1))
    train_idx_list, test_idx_list = [], []
    for b in np.unique(bins):
        b_idx = idx[bins == b]
        rng.shuffle(b_idx)
        n_test_b = int(round(len(b_idx) * test_ratio))
        test_idx_list.append(b_idx[:n_test_b])
        train_idx_list.append(b_idx[n_test_b:])
    train_idx = np.concatenate(train_idx_list)
    test_idx = np.concatenate(test_idx_list)
    rng.shuffle(train_idx); rng.shuffle(test_idx)
    return train_idx, test_idx


def standardize_train_test(X_train, X_test, eps=1e-6):
    mu = X_train.mean(axis=0, keepdims=True)
    std = X_train.std(axis=0, keepdims=True)
    std = np.maximum(std, eps)
    return (X_train - mu) / std, (X_test - mu) / std, mu, std


def make_loaders(X_train, y_train, X_test, y_test, batch_size=128):
    X_train_t = torch.from_numpy(X_train).float()
    y_train_t = torch.from_numpy(y_train).float().unsqueeze(1)  # (N,1)
    X_test_t  = torch.from_numpy(X_test).float()
    y_test_t  = torch.from_numpy(y_test).float().unsqueeze(1)

    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(TensorDataset(X_test_t,  y_test_t),  batch_size=batch_size, shuffle=False)
    return train_loader, test_loader


# =============================
# MAIN
# =============================
y_prob = build_prob_unbound_labels("dataset/completed_systems_sampling_0.txt", token_size=8, threshold_time=5000.0)
X, used_keys = load_tokenized_X_take_one("dataset/ICs_sampling_0_groupof8.hdf5", keys=None, take_token_idx=0)

# align
N = min(X.shape[0], y_prob.shape[0])
X = X[:N]
y_prob = y_prob[:N]

print("X:", X.shape, "y_prob:", y_prob.shape, "keys_used:", len(used_keys))

train_idx, test_idx = train_test_split_groups(X, y_prob, test_ratio=0.2, seed=42, stratify=True)

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y_prob[train_idx], y_prob[test_idx]

X_train, X_test, mu, std = standardize_train_test(X_train, X_test)

train_loader, test_loader = make_loaders(X_train, y_train, X_test, y_test, batch_size=128)

print("Train:", X_train.shape, y_train.shape, " Test:", X_test.shape, y_test.shape)
print("y_train prob stats:", float(y_train.min()), float(y_train.mean()), float(y_train.max()))
print("y_test  prob stats:", float(y_test.min()), float(y_test.mean()), float(y_test.max()))


X: (5000, 17) y_prob: (5000,) keys_used: 6
Train: (4000, 17) (4000,)  Test: (1000, 17) (1000,)
y_train prob stats: 0.0 0.35603123903274536 1.0
y_test  prob stats: 0.0 0.3636249899864197 1.0


## mlp

In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLPProb(nn.Module):
    """
    Predict probability p(unbound) in [0,1] from flattened initial condition features.
    """
    def __init__(self, in_dim: int, hidden=(512, 256, 128), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [
                nn.Linear(prev, h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            prev = h
        layers += [nn.Linear(prev, 1)]  # logits
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        logits = self.net(x)                 # (B,1)
        prob = torch.sigmoid(logits)         # (B,1) in [0,1]
        return prob


def train_mlp_prob(
    train_loader,
    test_loader,
    in_dim: int,
    device=None,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=30,
):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = MLPProb(in_dim=in_dim).to(device)

    # Since y is a probability in [0,1], MSE is a clean baseline.
    # You can also try: nn.BCELoss() (on prob) or BCEWithLogitsLoss (on logits).
    loss_fn = nn.MSELoss()

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    for ep in range(1, epochs + 1):
        # ---- train ----
        model.train()
        train_loss = 0.0
        n_train = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            pred = model(xb)                 # (B,1) prob
            loss = loss_fn(pred, yb)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            bs = xb.size(0)
            train_loss += loss.item() * bs
            n_train += bs

        train_loss /= max(n_train, 1)

        # ---- eval ----
        model.eval()
        test_loss = 0.0
        mae = 0.0
        n_test = 0

        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                pred = model(xb)
                loss = loss_fn(pred, yb)

                bs = xb.size(0)
                test_loss += loss.item() * bs
                mae += torch.abs(pred - yb).sum().item()
                n_test += bs

        test_loss /= max(n_test, 1)
        mae /= max(n_test, 1)

        print(f"epoch {ep:03d} | train_mse {train_loss:.6f} | test_mse {test_loss:.6f} | test_mae {mae:.6f}")

    return model

In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
in_dim = X_train.shape[1]
model = train_mlp_prob(train_loader, test_loader, in_dim=in_dim, device=device, epochs=200)

epoch 001 | train_mse 0.137086 | test_mse 0.129406 | test_mae 0.285584
epoch 002 | train_mse 0.115272 | test_mse 0.122252 | test_mae 0.269438
epoch 003 | train_mse 0.106203 | test_mse 0.112889 | test_mae 0.258007
epoch 004 | train_mse 0.100125 | test_mse 0.105662 | test_mae 0.254916
epoch 005 | train_mse 0.094088 | test_mse 0.101251 | test_mae 0.249221
epoch 006 | train_mse 0.090594 | test_mse 0.101773 | test_mae 0.240591
epoch 007 | train_mse 0.090056 | test_mse 0.099263 | test_mae 0.236586
epoch 008 | train_mse 0.085968 | test_mse 0.099922 | test_mae 0.239025
epoch 009 | train_mse 0.083172 | test_mse 0.096292 | test_mae 0.232763
epoch 010 | train_mse 0.078725 | test_mse 0.099102 | test_mae 0.230434
epoch 011 | train_mse 0.077564 | test_mse 0.100928 | test_mae 0.228821
epoch 012 | train_mse 0.077425 | test_mse 0.099731 | test_mae 0.230133
epoch 013 | train_mse 0.071976 | test_mse 0.100043 | test_mae 0.220667
epoch 014 | train_mse 0.068501 | test_mse 0.100674 | test_mae 0.226848
epoch 

## Transformer

In [38]:
import torch
import torch.nn as nn


class TransformerProb(nn.Module):
    """
    Predict probability p(unbound) in [0,1] from tokenized features.
    Input:  x of shape (B, T, in_dim)
    Output: prob of shape (B, 1)
    """
    def __init__(
        self,
        in_dim: int,
        n_tokens: int,
        d_model: int = 128,
        nhead: int = 8,
        num_layers: int = 4,
        dim_feedforward: int = 256,
        dropout: float = 0.2,
    ):
        super().__init__()

        self.in_proj = nn.Linear(in_dim, d_model)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_tokens + 1, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 1)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        # x: (B, T, in_dim)
        z = self.in_proj(x)  # (B, T, d_model)

        B, T, _ = z.shape
        cls = self.cls_token.expand(B, -1, -1)     # (B, 1, d_model)
        z = torch.cat([cls, z], dim=1)             # (B, 1+T, d_model)
        z = z + self.pos_embed[:, :T + 1, :]

        z = self.encoder(z)                        # (B, 1+T, d_model)
        z = self.norm(z[:, 0, :])                 # CLS token -> (B, d_model)

        logits = self.head(z)                     # (B, 1)
        prob = torch.sigmoid(logits)              # (B, 1)
        return prob


def train_transformer_prob(
    train_loader,
    test_loader,
    in_dim: int,
    device=None,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=30,
    n_tokens: int = 26,
    d_model: int = 128,
    nhead: int = 8,
    num_layers: int = 4,
    dim_feedforward: int = 256,
    dropout: float = 0.2,
):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = TransformerProb(
        in_dim=in_dim,
        n_tokens=n_tokens,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=dim_feedforward,
        dropout=dropout,
    ).to(device)

    loss_fn = nn.MSELoss()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    for ep in range(1, epochs + 1):
        # ---- train ----
        model.train()
        train_loss = 0.0
        n_train = 0

        for xb, yb in train_loader:
            xb = xb.to(device).float()   # expect (B, T, in_dim)
            yb = yb.to(device).float()

            pred = model(xb)             # (B,1)
            loss = loss_fn(pred, yb)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            bs = xb.size(0)
            train_loss += loss.item() * bs
            n_train += bs

        train_loss /= max(n_train, 1)

        # ---- eval ----
        model.eval()
        test_loss = 0.0
        mae = 0.0
        n_test = 0

        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(device).float()
                yb = yb.to(device).float()

                pred = model(xb)
                loss = loss_fn(pred, yb)

                bs = xb.size(0)
                test_loss += loss.item() * bs
                mae += torch.abs(pred - yb).sum().item()
                n_test += bs

        test_loss /= max(n_test, 1)
        mae /= max(n_test, 1)

        print(f"epoch {ep:03d} | train_mse {train_loss:.6f} | test_mse {test_loss:.6f} | test_mae {mae:.6f}")

    return model

In [ ]:
def make_token_loaders(X_train, y_train, X_test, y_test, batch_size=128):
    X_train_t = torch.from_numpy(X_train).float()   # (N, T, d)
    y_train_t = torch.from_numpy(y_train).float().unsqueeze(1)
    X_test_t  = torch.from_numpy(X_test).float()
    y_test_t  = torch.from_numpy(y_test).float().unsqueeze(1)

    print("X_train_t.shape: ", X_train_t.shape)
    print("y_train_t.shape: ", y_train_t.shape)

    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=batch_size, shuffle=False)
    return train_loader, test_loader

train_loader_transformer, test_loader_transformer = make_token_loaders(X_train, y_train, X_test, y_test, batch_size=128)

X_train_t.shape:  torch.Size([4000, 26, 3])
y_train_t.shape:  torch.Size([4000, 1])


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

n_tokens = X_train.shape[1]
in_dim = X_train.shape[2]

model = train_transformer_prob(
    train_loader_transformer,
    test_loader_transformer,
    in_dim=in_dim,
    n_tokens=n_tokens,
    device=device,
    epochs=200,
)

/tmp/ipykernel_67342/1996308098.py:37: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


epoch 001 | train_mse 0.181625 | test_mse 0.171027 | test_mae 0.364077
epoch 002 | train_mse 0.168173 | test_mse 0.159029 | test_mae 0.355609
epoch 003 | train_mse 0.153764 | test_mse 0.166029 | test_mae 0.366938
epoch 004 | train_mse 0.147436 | test_mse 0.133888 | test_mae 0.309885
epoch 005 | train_mse 0.136323 | test_mse 0.139332 | test_mae 0.292933
epoch 006 | train_mse 0.131523 | test_mse 0.126528 | test_mae 0.290292
epoch 007 | train_mse 0.122074 | test_mse 0.125715 | test_mae 0.282932
epoch 008 | train_mse 0.119049 | test_mse 0.125809 | test_mae 0.292150
epoch 009 | train_mse 0.119286 | test_mse 0.130568 | test_mae 0.276949
epoch 010 | train_mse 0.117109 | test_mse 0.122927 | test_mae 0.273595
epoch 011 | train_mse 0.115856 | test_mse 0.119607 | test_mae 0.272969
epoch 012 | train_mse 0.115414 | test_mse 0.125051 | test_mae 0.282333
epoch 013 | train_mse 0.112528 | test_mse 0.122371 | test_mae 0.268295
epoch 014 | train_mse 0.111199 | test_mse 0.122285 | test_mae 0.276269
epoch 

## XGBoost

In [49]:
# xgb_prob_regression_stable.py
# Stable XGBoost regressor for predicting probability in [0,1]

from __future__ import annotations
import numpy as np

try:
    import xgboost as xgb
except ImportError as e:
    raise ImportError("Missing dependency: xgboost. Install with: pip install xgboost") from e

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# ---------------------------
# Utilities
# ---------------------------
def loader_to_numpy(loader):
    """
    Convert a PyTorch DataLoader yielding (xb, yb) into numpy arrays.

    Expected:
        xb: (B, D)
        yb: (B, 1) or (B,)

    Returns:
        X: (N, D) float32
        y: (N,) float32
    """
    xs, ys = [], []
    for xb, yb in loader:
        xs.append(xb.detach().cpu().numpy())
        ys.append(yb.detach().cpu().numpy())

    X = np.concatenate(xs, axis=0).astype(np.float32)
    y = np.concatenate(ys, axis=0).reshape(-1).astype(np.float32)
    return X, y


def clip01(a: np.ndarray) -> np.ndarray:
    return np.clip(a, 0.0, 1.0)


def check_xy(X: np.ndarray, y: np.ndarray, name="data"):
    X = np.asarray(X)
    y = np.asarray(y)

    if X.ndim != 2:
        raise ValueError(f"{name}: X must be 2D (N, D), got shape {X.shape}")
    if y.ndim not in (1, 2):
        raise ValueError(f"{name}: y must be 1D or 2D, got shape {y.shape}")

    y = y.reshape(-1)
    if X.shape[0] != y.shape[0]:
        raise ValueError(f"{name}: X has {X.shape[0]} rows but y has {y.shape[0]} rows")

    if not np.isfinite(X).all():
        raise ValueError(f"{name}: X contains NaN or inf")
    if not np.isfinite(y).all():
        raise ValueError(f"{name}: y contains NaN or inf")

    return X.astype(np.float32), y.astype(np.float32)


# ---------------------------
# Model
# ---------------------------
class XGBProbRegressor:
    """
    Predict probability p in [0,1] by regression.

    Input:
        X: (N, D)

    Output:
        pred: (N, 1), clipped into [0,1]
    """
    def __init__(
        self,
        n_estimators: int = 500,
        learning_rate: float = 0.05,
        max_depth: int = 6,
        subsample: float = 0.8,
        colsample_bytree: float = 0.8,
        reg_lambda: float = 1.0,
        reg_alpha: float = 0.0,
        min_child_weight: float = 1.0,
        gamma: float = 0.0,
        random_state: int = 42,
        tree_method: str = "hist",
        n_jobs: int = -1,
    ):
        self.model = xgb.XGBRegressor(
            objective="reg:squarederror",
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            reg_lambda=reg_lambda,
            reg_alpha=reg_alpha,
            min_child_weight=min_child_weight,
            gamma=gamma,
            random_state=random_state,
            tree_method=tree_method,
            n_jobs=n_jobs,
        )

    def fit(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_val: np.ndarray | None = None,
        y_val: np.ndarray | None = None,
        verbose: bool = True,
    ):
        X_train, y_train = check_xy(X_train, y_train, name="train")

        eval_set = None
        if X_val is not None and y_val is not None:
            X_val, y_val = check_xy(X_val, y_val, name="val")
            eval_set = [(X_train, y_train), (X_val, y_val)]

        self.model.fit(
            X_train,
            y_train,
            eval_set=eval_set,
            verbose=verbose,
        )
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=np.float32)
        if X.ndim != 2:
            raise ValueError(f"X must be 2D (N, D), got shape {X.shape}")

        pred = self.model.predict(X)
        pred = clip01(pred)
        return pred.reshape(-1, 1)


# ---------------------------
# Evaluation
# ---------------------------
def evaluate_prob_regression(y_true: np.ndarray, y_pred: np.ndarray, prefix=""):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"{prefix}mse {mse:.6f} | mae {mae:.6f} | r2 {r2:.6f}")
    return {"mse": mse, "mae": mae, "r2": r2}


# ---------------------------
# Training from arrays
# ---------------------------
def train_xgb_prob_from_arrays(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    X_val: np.ndarray | None = None,
    y_val: np.ndarray | None = None,
    **xgb_kwargs,
):
    X_train, y_train = check_xy(X_train, y_train, name="train")
    X_test, y_test = check_xy(X_test, y_test, name="test")

    if X_val is not None and y_val is not None:
        X_val, y_val = check_xy(X_val, y_val, name="val")

    model = XGBProbRegressor(**xgb_kwargs)
    model.fit(
        X_train,
        y_train,
        X_val=X_val,
        y_val=y_val,
        verbose=True,
    )

    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    print("XGB | TRAIN:", end=" ")
    evaluate_prob_regression(y_train, pred_train)

    print("XGB | TEST :", end=" ")
    evaluate_prob_regression(y_test, pred_test)

    return model


# ---------------------------
# Training from PyTorch loaders
# ---------------------------
def train_xgb_prob_from_loaders(
    train_loader,
    test_loader,
    val_loader=None,
    **xgb_kwargs,
):
    X_train, y_train = loader_to_numpy(train_loader)
    X_test, y_test = loader_to_numpy(test_loader)

    if val_loader is not None:
        X_val, y_val = loader_to_numpy(val_loader)
    else:
        X_val, y_val = None, None

    return train_xgb_prob_from_arrays(
        X_train, y_train,
        X_test, y_test,
        X_val=X_val, y_val=y_val,
        **xgb_kwargs,
    )

In [ ]:
model = train_xgb_prob_from_loaders(
    train_loader,
    test_loader,
    val_loader=None,      
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
)

XGB | TRAIN: mse 0.005598 | mae 0.053982 | r2 0.967315
XGB | TEST : mse 0.098321 | mae 0.243547 | r2 0.427884


In [ ]:
import xgboost
print(xgboost.__version__)

3.2.0
